In [0]:
from functools import reduce
from pyspark.sql import functions as F

# (nome da demonstração, tabela bronze). DFC vem por 2 métodos: indireto (MI) e direto (MD)
fontes = [
    ("DRE", "dfp_dre_con"),
    ("BPA", "dfp_bpa_con"),     # balanço: ativo
    ("BPP", "dfp_bpp_con"),     # balanço: passivo + PL
    ("DFC", "dfp_dfc_mi_con"),
    ("DFC", "dfp_dfc_md_con"),
]

dfs = [spark.table(f"cvm.bronze.{tabela}").withColumn("demonstracao", F.lit(nome))
       for nome, tabela in fontes]

# unionByName: empilha as tabelas casando as colunas pelo NOME (o balanço não tem DT_INI_EXERC, por exemplo)
bruto = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
bruto.createOrReplaceTempView("vw_demonstracoes_bruto")
print(f"{bruto.count():,} linhas brutas")

In [0]:
%sql
CREATE OR REPLACE TABLE cvm.silver.demonstracoes
COMMENT 'Demonstrações consolidadas: exercício corrente, última versão, valores em reais'
AS
SELECT
  regexp_replace(CNPJ_CIA, '[^0-9]', '')         AS cnpj,
  CAST(CD_CVM AS INT)                            AS cd_cvm,
  trim(DENOM_CIA)                                AS empresa,
  demonstracao,
  CAST(DT_FIM_EXERC AS DATE)                     AS dt_fim_exerc,
  year(CAST(DT_FIM_EXERC AS DATE))               AS ano,
  CAST(VERSAO AS INT)                            AS versao,
  CD_CONTA                                       AS cd_conta,
  trim(DS_CONTA)                                 AS ds_conta,
  ST_CONTA_FIXA = 'S'                            AS conta_fixa,
  CAST(VL_CONTA AS DECIMAL(24,2))
    * CASE WHEN ESCALA_MOEDA = 'MIL' THEN 1000 ELSE 1 END AS valor_reais,
  _arquivo_origem,
  current_timestamp()                            AS _processado_em
FROM vw_demonstracoes_bruto
WHERE ORDEM_EXERC = 'ÚLTIMO'                     -- descarta o ano comparativo
QUALIFY CAST(VERSAO AS INT) =                    -- fica com a última versão entregue
        max(CAST(VERSAO AS INT)) OVER (PARTITION BY CNPJ_CIA, DT_FIM_EXERC, demonstracao)

In [0]:
%sql
SELECT demonstracao, count(*) AS linhas, count(DISTINCT cnpj) AS empresas
FROM cvm.silver.demonstracoes
GROUP BY demonstracao

In [0]:
%sql
CREATE OR REPLACE TABLE cvm.silver.empresas AS
SELECT
  regexp_replace(CNPJ_CIA, '[^0-9]', '')                 AS cnpj,
  CAST(CD_CVM AS INT)                                    AS cd_cvm,
  trim(DENOM_SOCIAL)                                     AS razao_social,
  trim(DENOM_COMERC)                                     AS nome_comercial,
  coalesce(nullif(trim(SETOR_ATIV), ''), 'Não informado') AS setor,
  SIT                                                    AS situacao
FROM cvm.bronze.cadastro_cia
-- uma empresa pode ter mais de um registro: fica com o ativo mais recente
QUALIFY row_number() OVER (PARTITION BY CNPJ_CIA
                           ORDER BY (SIT = 'ATIVO') DESC, DT_REG DESC) = 1

In [0]:
%sql
SELECT empresa, ano, ds_conta, format_number(valor_reais, 0) AS valor
FROM cvm.silver.demonstracoes
WHERE empresa LIKE '%AMBEV%' AND demonstracao = 'DRE' AND cd_conta IN ('3.01', '3.11')
ORDER BY ano, cd_conta

In [0]:
%sql
-- Ativo total (conta 1) tem que ser igual a Passivo + PL (conta 2). Se vier vazio, PASSOU.
SELECT a.empresa, a.ano,
       a.valor_reais AS ativo, p.valor_reais AS passivo_mais_pl,
       a.valor_reais - p.valor_reais AS diferenca
FROM cvm.silver.demonstracoes a
JOIN cvm.silver.demonstracoes p
  ON p.cnpj = a.cnpj AND p.dt_fim_exerc = a.dt_fim_exerc
 AND p.demonstracao = 'BPP' AND p.cd_conta = '2'
WHERE a.demonstracao = 'BPA' AND a.cd_conta = '1'
  AND abs(a.valor_reais - p.valor_reais) > greatest(1000, abs(a.valor_reais) * 0.0001)

In [0]:
# Trava de qualidade: se algum balanço não fechar, o job FALHA e a gold não é atualizada
divergencias = spark.sql("""
    SELECT a.cnpj
    FROM cvm.silver.demonstracoes a
    JOIN cvm.silver.demonstracoes p
      ON p.cnpj = a.cnpj AND p.dt_fim_exerc = a.dt_fim_exerc
     AND p.demonstracao = 'BPP' AND p.cd_conta = '2'
    WHERE a.demonstracao = 'BPA' AND a.cd_conta = '1'
      AND abs(a.valor_reais - p.valor_reais) > greatest(1000, abs(a.valor_reais) * 0.0001)
""").count()

assert divergencias == 0, f"Teste de qualidade falhou: {divergencias} balanços não fecham"
print("Teste de fechamento do balanço: OK")